In [1]:
# import libraries and other configurations
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import warnings
from scipy import stats
from dython.nominal import associations

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

sns.set_theme(style="whitegrid")

warnings.filterwarnings("ignore")

RANDOM_STATE = 121

In [2]:
# Load cleaned dataset
df = pd.read_csv(
    "../data/processed/cleaned_donor_data.csv",
)

df.columns.tolist()

['donor_unique_id',
 'donor_postal_code',
 'donor_age',
 'gender_identity',
 'is_member_flag',
 'is_alumnus_flag',
 'is_parent_flag',
 'has_involvement_flag',
 'preferred_address_type',
 'has_email_flag',
 'consecutive_donor_years',
 'last_fiscal_year_donation',
 'donation_2_fiscal_years_ago',
 'donation_3_fiscal_years_ago',
 'donation_4_fiscal_years_ago',
 'donation_5_fiscal_years_ago',
 'current_fiscal_year_donation',
 'cumulative_donation_amount',
 'donor_indicator_flag']

In [3]:
# Verify dataset loaded correctly
print(f"Dataset Shape: {df.shape}")
df.sample(10, random_state=RANDOM_STATE)

Dataset Shape: (34403, 19)


,donor_unique_id,donor_postal_code,donor_age,gender_identity,is_member_flag,is_alumnus_flag,is_parent_flag,has_involvement_flag,preferred_address_type,has_email_flag,consecutive_donor_years,last_fiscal_year_donation,donation_2_fiscal_years_ago,donation_3_fiscal_years_ago,donation_4_fiscal_years_ago,donation_5_fiscal_years_ago,current_fiscal_year_donation,cumulative_donation_amount,donor_indicator_flag
33267,33371,42301.0,46,Female,0,0,0,0,Home,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
26595,26683,45620.0,42,Female,0,0,0,0,Home,0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
16653,16704,33427.0,28,Female,0,0,0,0,Home,0,0,0.0,120.0,0.0,0.0,0.0,0.0,120.0,1
21428,21498,54131.0,33,Female,0,0,0,0,Home,0,0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1
16873,16924,90265.0,33,Male,0,1,0,0,Home,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
19919,19982,54459.0,32,Female,0,0,0,0,Home,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
15478,15527,64686.0,42,Male,0,0,0,0,Home,0,8,0.0,0.0,10.0,0.0,0.0,0.0,9680.0,1
591,596,12067.0,42,Female,0,0,1,0,Home,0,0,0.0,100.0,1.0,0.0,0.0,0.0,101.0,1
1474,1483,90265.0,42,Female,0,1,0,0,Home,1,0,0.0,0.0,0.0,0.0,0.0,0.0,479.0,1
22846,22922,45856.0,36,Female,0,0,0,0,Home,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [4]:
# Verify dataset data types
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34403 entries, 0 to 34402
Data columns (total 19 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   donor_unique_id               34403 non-null  int64  
 1   donor_postal_code             34312 non-null  float64
 2   donor_age                     34403 non-null  int64  
 3   gender_identity               33912 non-null  str    
 4   is_member_flag                34403 non-null  int64  
 5   is_alumnus_flag               34403 non-null  int64  
 6   is_parent_flag                34403 non-null  int64  
 7   has_involvement_flag          34403 non-null  int64  
 8   preferred_address_type        30370 non-null  str    
 9   has_email_flag                34403 non-null  int64  
 10  consecutive_donor_years       34403 non-null  int64  
 11  last_fiscal_year_donation     34403 non-null  float64
 12  donation_2_fiscal_years_ago   34403 non-null  float64
 13  donation_3_f

In [5]:
# Data types summary
dtype_summary = (
    df.dtypes
      .astype(str)
      .value_counts()
      .reset_index()
      .style.hide(axis='index')
)

dtype_summary.columns = ["Data Type", "Count"]
dtype_summary

index,count
int64,9
float64,8
str,2


In [6]:
# Verify dataset missing values
missing_summary = (
    df.isna()
      .sum()
      .to_frame("Missing Count")
)

missing_summary["Missing %"] = (
    missing_summary["Missing Count"] / len(df) * 100
)

missing_summary = (
    missing_summary
      .sort_values("Missing Count", ascending=False)
)

missing_summary

,Missing Count,Missing %
preferred_address_type,4033,11.722815
gender_identity,491,1.427201
donor_postal_code,91,0.264512
donor_unique_id,0,0.000000
last_fiscal_year_donation,0,0.000000
cumulative_donation_amount,0,0.000000
current_fiscal_year_donation,0,0.000000
donation_5_fiscal_years_ago,0,0.000000
donation_4_fiscal_years_ago,0,0.000000
donation_3_fiscal_years_ago,0,0.000000


## Dataset Overview

The finalized cleaned analytical dataset was successfully loaded and verified for Dython analysis. The dataset contains **34,403 donor records** and **19 features**, consistent with the dataset used throughout Phase 2. Data types and missing values match expectations from the data cleaning process, and all required libraries were successfully imported. The environment is now prepared for mixed-type association analysis.

In [7]:
# Define feature groups
numeric_features = [
    "donor_age",
    "consecutive_donor_years",
    "last_fiscal_year_donation",
    "donation_2_fiscal_years_ago",
    "donation_3_fiscal_years_ago",
    "donation_4_fiscal_years_ago",
    "donation_5_fiscal_years_ago",
    "current_fiscal_year_donation",
    "cumulative_donation_amount"
]

categorical_features = [
    "gender_identity",
    "preferred_address_type"
]

binary_features = [
    "is_alumnus_flag",
    "is_parent_flag",
    "has_involvement_flag",
    "has_email_flag"
]

target_feature = [
    "donor_indicator_flag"
]

In [8]:
# Variable classification summary
association_features = (
    numeric_features
    + categorical_features
    + binary_features
    + target_feature
)

variable_summary = pd.DataFrame({
    "Variable": association_features,
    "Category": (
        ["Numeric"] * len(numeric_features)
        + ["Categorical"] * len(categorical_features)
        + ["Binary"] * len(binary_features)
        + ["Target"]
    )
})

variable_summary

,Variable,Category
0,donor_age,Numeric
1,consecutive_donor_years,Numeric
2,last_fiscal_year_donation,Numeric
3,donation_2_fiscal_years_ago,Numeric
4,donation_3_fiscal_years_ago,Numeric
5,donation_4_fiscal_years_ago,Numeric
6,donation_5_fiscal_years_ago,Numeric
7,current_fiscal_year_donation,Numeric
8,cumulative_donation_amount,Numeric
9,gender_identity,Categorical


In [9]:
# Verify selected features exist in the dataset
missing_features = [col for col in association_features if col not in df.columns]

if missing_features:
    print("Missing features:")
    print(missing_features)
else:
    print("All selected features are present in the dataset.")

All selected features are present in the dataset.


In [10]:
# Create analysis dataset for dython analysis
association_df = df[association_features].copy()

print(f"Association Dataset Shape: {association_df.shape}")

association_df.sample(10, random_state=RANDOM_STATE)

Association Dataset Shape: (34403, 16)


,donor_age,consecutive_donor_years,last_fiscal_year_donation,donation_2_fiscal_years_ago,donation_3_fiscal_years_ago,donation_4_fiscal_years_ago,donation_5_fiscal_years_ago,current_fiscal_year_donation,cumulative_donation_amount,gender_identity,preferred_address_type,is_alumnus_flag,is_parent_flag,has_involvement_flag,has_email_flag,donor_indicator_flag
33267,46,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Female,Home,0,0,0,0,0
26595,42,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Female,Home,0,0,0,0,0
16653,28,0,0.0,120.0,0.0,0.0,0.0,0.0,120.0,Female,Home,0,0,0,0,1
21428,33,0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,Female,Home,0,0,0,0,1
16873,33,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Male,Home,1,0,0,0,0
19919,32,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Female,Home,0,0,0,0,0
15478,42,8,0.0,0.0,10.0,0.0,0.0,0.0,9680.0,Male,Home,0,0,0,0,1
591,42,0,0.0,100.0,1.0,0.0,0.0,0.0,101.0,Female,Home,0,1,0,0,1
1474,42,0,0.0,0.0,0.0,0.0,0.0,0.0,479.0,Female,Home,1,0,0,1,1
22846,36,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Female,Home,0,0,0,0,0


In [11]:
# Verify analysis dataset
association_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34403 entries, 0 to 34402
Data columns (total 16 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   donor_age                     34403 non-null  int64  
 1   consecutive_donor_years       34403 non-null  int64  
 2   last_fiscal_year_donation     34403 non-null  float64
 3   donation_2_fiscal_years_ago   34403 non-null  float64
 4   donation_3_fiscal_years_ago   34403 non-null  float64
 5   donation_4_fiscal_years_ago   34403 non-null  float64
 6   donation_5_fiscal_years_ago   34403 non-null  float64
 7   current_fiscal_year_donation  34403 non-null  float64
 8   cumulative_donation_amount    34403 non-null  float64
 9   gender_identity               33912 non-null  str    
 10  preferred_address_type        30370 non-null  str    
 11  is_alumnus_flag               34403 non-null  int64  
 12  is_parent_flag                34403 non-null  int64  
 13  has_involvem

In [12]:
# Verify analysis dataset
association_df.nunique().sort_values()

is_alumnus_flag                    2
is_parent_flag                     2
has_involvement_flag               2
has_email_flag                     2
donor_indicator_flag               2
gender_identity                    3
preferred_address_type             4
consecutive_donor_years           33
donor_age                         88
current_fiscal_year_donation     165
donation_4_fiscal_years_ago      177
donation_5_fiscal_years_ago      182
last_fiscal_year_donation        188
donation_3_fiscal_years_ago      188
donation_2_fiscal_years_ago      189
cumulative_donation_amount      1592
dtype: int64